In [1]:
import os
import time
import csv
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

# Local imports from your repository
from utils import custom_ap_scorer, sklearn_custom_ap_scorer, sem_abrv_to_full
from machine_learning.svm.python.ranksvm import RankSVM

# Ensure artifacts directory exists
ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

In [2]:
# ==========================================
# 1. LOAD & FIX DATA
# ==========================================
print("Loading train and test datasets...")
train_df = pd.read_csv("train_df.csv", index_col=0)
test_df = pd.read_csv("test_df.csv", index_col=0)

# --- CRITICAL FIX: INVERT TEST LABELS ONLY ---
print("Applying label inversion fix to test data...")
test_df['target'] = 1 - test_df['target']

feature_names = train_df.columns.tolist()
feature_names.remove('label')
feature_names.remove('target')

X = train_df[feature_names].values
Y = train_df["target"].values

X_test = test_df[feature_names].values
Y_test = test_df["target"].values

print(f"Training on {X.shape[0]} nodes with {X.shape[1]} features.")
print(f"Testing on {X_test.shape[0]} nodes.")
print(f"Test Class Distribution: {np.sum(Y_test == 1)} Useful (1), {np.sum(Y_test == 0)} Not Useful (0)")

Loading train and test datasets...
Applying label inversion fix to test data...
Training on 1055 nodes with 181 features.
Testing on 76 nodes.
Test Class Distribution: 59 Useful (1), 17 Not Useful (0)


In [ ]:
# ==========================================
# 2. HYPERPARAMETER TUNING (GridSearch)
# ==========================================
# Set RUN_GRID_SEARCH = False to run in Fast Mode using C = 0.01
RUN_GRID_SEARCH = False

if RUN_GRID_SEARCH:
    print("\nRunning 10-Fold GridSearch for RankSVM...")
    C_range = np.logspace(-2, 2, 5)
    hyperparameters = dict(C=C_range, random_state=[2020])
    
    grid_search = GridSearchCV(
        RankSVM(), 
        hyperparameters, 
        cv=10, 
        verbose=0, 
        scoring=sklearn_custom_ap_scorer
    )
    
    start_time = time.time()
    grid_search.fit(X, Y)
    print(f"GridSearch finished in {time.time() - start_time:.1f} seconds")
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV AP score: {grid_search.best_score_:.4f}")
    
    best_params = grid_search.best_params_
else:
    print("\nFast Mode: Using thesis optimal parameters...")
    best_params = {'C': 0.01, 'random_state': 2020}


Running 10-Fold GridSearch for RankSVM...
GridSearch finished in 15026.1 seconds
Best parameters: {'C': 0.01, 'random_state': 2020}
Best CV AP score: 0.6994


In [4]:
# ==========================================
# 3. FIT FINAL MODEL & EVALUATE
# ==========================================
print("\nTraining Final RankSVM model...")
rank_svm = RankSVM(**best_params)
rank_svm.fit(X, Y)

Y_rank_svm_predict = rank_svm.predict(X_test)
ap_score = custom_ap_scorer(Y_test, Y_rank_svm_predict)

print("\n" + "="*50)
print(f"FINAL RANKSVM AVERAGE PRECISION: {ap_score:.4f}")
print("="*50)

# Save model coefficients
joblib.dump(rank_svm.coefs, f"{ARTIFACT_DIR}/rankSVMcoefs.joblib")


Training Final RankSVM model...

FINAL RANKSVM AVERAGE PRECISION: 0.9479


['artifacts/rankSVMcoefs.joblib']

In [5]:
# ==========================================
# 4. FEATURE WEIGHT & ERROR ANALYSIS
# ==========================================
print("\nTop 25 Most Influential Features:")
unabrv_feature_name = [
    sem_abrv_to_full.get(feat, feat) for feat in feature_names
]

features_coef = list(zip(unabrv_feature_name, rank_svm.coefs.tolist()))
features_coef.sort(key=lambda x: abs(x[1]), reverse=True)

header = f"{'Feature Name':40} | {'Model Coef':10}"
print(header)
print("-" * len(header))
for feat_name, coef in features_coef[:25]:
    print(f"{feat_name:40} | {coef:+2.5f}")


Top 25 Most Influential Features:
Feature Name                             | Model Coef
-----------------------------------------------------
feat_7                                   | +6.93966
feat_74                                  | -4.34124
feat_156                                 | +4.28429
feat_52                                  | -4.27101
feat_115                                 | -3.65448
feat_29                                  | -3.56571
feat_40                                  | -3.28064
feat_148                                 | +3.27682
feat_36                                  | +2.90981
feat_66                                  | -2.89102
feat_121                                 | -2.75258
feat_54                                  | -2.61510
feat_26                                  | +2.59963
feat_157                                 | +2.57153
feat_135                                 | -2.50574
feat_129                                 | -2.37796
feat_139                 

In [6]:
# ==========================================
# 5. EXPORT FINAL RANKINGS TABLE
# ==========================================
print("\nExporting final_model_eval.csv...")
rankings = zip(test_df.index.tolist(), Y_rank_svm_predict.tolist())
column_names = ['#', 'cui', 'label', 'ranking', 'target'] + unabrv_feature_name
cui_to_x_test = {cui: X_test[ii, :].tolist() for ii, cui in enumerate(test_df.index)}

with open('final_model_eval.csv', 'w', newline='', encoding='utf-8') as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(column_names)
    writer.writerow(['', '', 'Model Coefficients', '', ''] + rank_svm.coefs.tolist())
    for ii, (cui, ranking) in enumerate(sorted(rankings, key=lambda x: x[1], reverse=True), start=1):
        true_label = "keep" if test_df['target'][cui] == 1 else "discard"
        writer.writerow([ii, cui, test_df['label'][cui], ranking, true_label] + cui_to_x_test[cui])

print("RankSVM evaluation and ranking export complete.")


Exporting final_model_eval.csv...
RankSVM evaluation and ranking export complete.
